# Makemore Lesson 04

Live notes and code while following Andrej Karpathy's makemore Part 4: **Becoming a Backprop Ninja**.

## Session Goals

- Start from the same boilerplate Karpathy uses before the manual-backprop exercises.
- Keep the forward pass broken into small named tensors.
- Use PyTorch gradients only as the reference check.
- Add short notes only when a concept is confusing or important.


In [ ]:
# there is no change in the first several cells from last lecture

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline


## Data

In [3]:
# read in all the words
words = open('../data/names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])


32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)


{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [5]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):
  X, Y = [], []

  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [ ]:
# ok boilerplate done, now we get to the action:

In [6]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')


In [7]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.1

# Note: many parameters are initialized in non-standard ways
# because all-zero/simple init can mask an incorrect backward pass implementation.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True


4137


In [45]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y


In [59]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if using (1.0 / counts_sum), backprop won't be bit exact
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
          bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
          embcat, emb]:
  t.retain_grad()
loss.backward()
loss


tensor(3.6271, grad_fn=<NegBackward0>)

In [72]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

# loss = -logprobs[range(n), Yb].mean()
# loss = -(a+b+c) / 3
# dloss / da = -1/3
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
cmp('logprobs', dlogprobs, logprobs)

# logprobs = probs.log()
dprobs = (1.0 / probs) * dlogprobs
cmp('probs', dprobs, probs)

#probs = counts * counts_sum_inv
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
dcounts = counts_sum_inv * dprobs

#counts_sum_inv = counts_sum**-1
dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, counts_sum)

# counts_sum = counts.sum(1, keepdims=True)
dcounts += torch.ones_like(counts) * dcounts_sum 
cmp('counts', dcounts, counts)

# counts = norm_logits.exp()
dnorm_logits = counts * dcounts
cmp('norm_logits', dnorm_logits, norm_logits)

#norm_logits = logits - logit_maxes
dlogits = dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)
cmp('logit_maxes', dlogit_maxes, logit_maxes)


#logit_maxes = logits.max(1, keepdim=True).values
dlogits += F.one_hot(logits.max(1).indices, num_classes= logits.shape[1]) * dlogit_maxes
cmp('logits', dlogits, logits)


#logits = h @ W2 + b2
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
# the dimension will work too
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)

#h = torch.tanh(hpreact)
dhpreact = (1.0 - h**2) * dh
cmp('hpreact', dhpreact, hpreact)

#hpreact = bngain * bnraw + bnbias
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)

#bnraw = bndiff * bnvar_inv
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)

#bnvar_inv = (bnvar + 1e-5)**-0.5
dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv


#bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) 
dbndiff2 = (1/(n-1))*torch.ones_like(bndiff2) * dbnvar

#bndiff2 = bndiff**2
dbndiff += (2 * bndiff) * dbndiff2

#bndiff = hprebn - bnmeani
dhprebn = dbndiff
dbnmeani = (-dbndiff).sum(0, keepdim=True)

#bnmeani = 1/n*hprebn.sum(0, keepdim=True)
dhprebn += 1/n * (torch.ones_like(hprebn) * dbnmeani) 

#hprebn = embcat @ W1 + b1
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

#embcat = emb.view(emb.shape[0], -1)
demb = dembcat.view(emb.shape)

#emb = C[Xb]
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k,j]
        dC[ix] += demb[k,j]

cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)


cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

## Exercise 2 · cross_entropy backward 一步推导

目标：不要再沿着 `logit_maxes -> norm_logits -> counts -> probs -> logprobs` 一步步反推，而是直接从 cross entropy 推出 `dlogits`。这里先放起手 block，答案你听课时自己补。


In [ ]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge, look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and write it out.

# forward pass
# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())


In [73]:
# backward pass

# TODO: derive dlogits directly from softmax + cross entropy
dlogits = F.softmax(logits, 1)
dlogits[range(n), Yb] -= 1
dlogits /= n

cmp('logits', dlogits, logits)


logits          | exact: False | approximate: True  | maxdiff: 5.587935447692871e-09


In [ ]:
# optional intuition / shape checks for Exercise 2
# logits.shape, Yb.shape
# F.softmax(logits, 1)[0]
# dlogits[0] * n
# dlogits[0].sum()
# plt.figure(figsize=(4, 4)); plt.imshow(dlogits.detach())


## Exercise 3 · BatchNorm backward 一步推导

目标：不要再沿着 `bndiff -> bndiff2 -> bnvar -> bnvar_inv -> bnraw` 一步步反推，而是把 BatchNorm 的 backward 化简成一个更紧凑的 `dhprebn`。这里也只放起手 block。


In [75]:
# Exercise 3: backprop through batchnorm but all in one go

# backward pass
# Inputs you will use from earlier derivation / forward pass:
# dhpreact, bngain, bnvar_inv, bnraw, bndiff, n

# TODO: derive compact dhprebn
dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))

cmp('hprebn', dhprebn, hprebn)


hprebn          | exact: False | approximate: True  | maxdiff: 4.656612873077393e-10


In [ ]:
# optional shape checks for Exercise 3
# dhprebn.shape, bngain.shape, bnvar_inv.shape
# dbnraw.shape, dbnraw.sum(0).shape


## Exercise 4 · putting it all together

目标：把前面手写的 backward 组合进训练循环。这里先放训练循环起手 block，不填完整答案。


In [ ]:
# Exercise 4: putting it all together!
# Re-initialize parameters if you want a clean run, then train using manual backward.

max_steps = 200000
batch_size = 32
lossi = []

# TODO: optional clean re-initialization of C, W1, b1, W2, b2, bngain, bnbias
# TODO: parameters = [C, W1, b1, W2, b2, bngain, bnbias]
# TODO: for p in parameters: p.requires_grad = True

for i in range(max_steps):

  # minibatch
  # ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  # Xb, Yb = Xtr[ix], Ytr[ix]

  # forward pass
  # TODO: compute emb, embcat, hprebn, bn stats, h, logits, loss

  # backward pass
  # TODO: use simplified dlogits from Exercise 2
  # TODO: use simplified dhprebn from Exercise 3
  # TODO: compute gradients for C, W1, b1, W2, b2, bngain, bnbias

  # update
  # lr = 0.1 if i < 100000 else 0.01
  # for p, grad in zip(parameters, grads):
  #   p.data += -lr * grad

  # track stats
  # if i % 10000 == 0:
  #   print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  # lossi.append(loss.log10().item())

  pass


In [ ]:
# useful for checking your gradients while building Exercise 4
# for p, grad in zip(parameters, grads):
#   cmp(str(tuple(p.shape)), grad, p)


In [ ]:
# calibrate the batch norm at the end of training
# with torch.no_grad():
#   emb = C[Xtr]
#   embcat = emb.view(emb.shape[0], -1)
#   hpreact = embcat @ W1 + b1
#   bnmean = hpreact.mean(0, keepdim=True)
#   bnvar = hpreact.var(0, keepdim=True, unbiased=True)

# evaluate train and val loss
# @torch.no_grad()
# def split_loss(split):
#   x, y = {
#     'train': (Xtr, Ytr),
#     'val': (Xdev, Ydev),
#     'test': (Xte, Yte),
#   }[split]
#   emb = C[x]
#   embcat = emb.view(emb.shape[0], -1)
#   hpreact = embcat @ W1 + b1
#   hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
#   h = torch.tanh(hpreact)
#   logits = h @ W2 + b2
#   loss = F.cross_entropy(logits, y)
#   print(split, loss.item())

# split_loss('train')
# split_loss('val')


## Notes

- 


### Why Track Max Gradient Instead of Only Loss?

Karpathy's point: `loss` tells us the batch-level objective, but it can average away rare extreme backward signals. For debugging training health, we also want to know whether any parameter received a dangerously large gradient.

```python
p.grad.abs().max()
```

This asks: *does this layer have any gradient value that is unusually large?*

A histogram or average can make outliers look unimportant because most gradients may sit near zero. Example: 99.9% of gradients are around `-0.001 ~ 0.001`, but a few values are `0.2`. Those few values may barely show up visually, but they can dominate individual parameter updates.

So:

- `loss`: overall model error for the batch
- gradient distribution: typical backward signal
- `grad.abs().max()`: whether extreme outlier gradients exist

The outliers are "ignored" only in the sense that aggregate views can hide them, not because PyTorch ignores them during optimization.


### Leaf Tensor and `retain_grad()`

A **leaf tensor** is a tensor that is created directly by the user and is not the result of another operation. In this notebook, the learnable parameters are leaf tensors:

```python
C, W1, b1, W2, b2, bngain, bnbias
```

These are the tensors we update during training, so PyTorch keeps their `.grad` after:

```python
loss.backward()
```

But intermediate tensors are not leaf tensors:

```python
logprobs, probs, counts, logits, h, hpreact, emb
```

They are created during the forward pass. By default, PyTorch uses their gradients internally for backprop, but does **not** keep their `.grad` values after backward.

Part 4 needs those intermediate gradients because we compare our manual backward results against PyTorch's answer:

```python
cmp('logprobs', dlogprobs, logprobs)
```

So we call:

```python
t.retain_grad()
```

on each intermediate tensor we want to inspect. In short: `retain_grad()` tells PyTorch to keep `.grad` for non-leaf intermediate tensors, so `cmp` can check our handwritten backward pass.


### softmax 数值稳定：`logit_maxes` / `norm_logits`

这段是在 softmax 前做数值稳定处理：

```python
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
```

假设：

```python
logits.shape == (32, 27)
```

含义是：32 个样本，每个样本有 27 个类别的 logit。`logits.max(1, keepdim=True)` 是沿着第 1 维，也就是每一行的 27 个类别，找出这一行最大的 logit。

例子：

```text
logits = [2.0, 5.0, 1.0, -3.0]
logit_max = 5.0
norm_logits = [-3.0, 0.0, -4.0, -8.0]
```

最大值会变成 0，其他值都小于等于 0。这样后面做：

```python
counts = norm_logits.exp()
```

不会因为 `exp(很大的数)` 而数值爆炸。

为什么 softmax 结果不变？因为每一行都减掉同一个数 `m`：

```text
exp(logit_i - m) / sum(exp(logit_j - m))
= [exp(logit_i) / exp(m)] / [sum(exp(logit_j)) / exp(m)]
= exp(logit_i) / sum(exp(logit_j))
```

分子和分母里的 `exp(m)` 会抵消掉，所以概率不变。

`keepdim=True` 的作用是保留 shape：

```python
logits.shape       # (32, 27)
logit_maxes.shape # (32, 1)
```

这样 `logits - logit_maxes` 可以 broadcasting：每一行都减掉自己那一行的最大值。

一句话：`norm_logits` 让每一行最大 logit 变成 0，避免 `exp` 爆炸，同时不改变 softmax 概率。


### broadcasting backward：为什么要 `sum(1, keepdim=True)`

这里对应 forward：

```python
probs = counts * counts_sum_inv
```

shape 是：

```python
counts.shape         # (32, 27)
counts_sum_inv.shape # (32, 1)
probs.shape          # (32, 27)
```

可以把 `counts` 看成一张 32 行、27 列的表：

```text
              .       a       b      ...      z
样本0      c00     c01     c02     ...    c026
样本1      c10     c11     c12     ...    c126
...
样本31     c310    c311    c312    ...    c3126
```

`counts_sum_inv` 是每一行只有一个数：

```text
样本0      s0_inv
样本1      s1_inv
...
样本31     s31_inv
```

forward 时，`s0_inv` 会被 broadcast 给第 0 行的 27 个类别：

```text
probs[0] = [c00*s0_inv, c01*s0_inv, c02*s0_inv, ..., c026*s0_inv]
```

所以 backward 时，`s0_inv` 会从第 0 行的 27 个 `probs` 收到梯度贡献。

如果从右边传回来的梯度是：

```text
dprobs[0] = [dp00, dp01, dp02, ..., dp026]
```

那么每个位置给 `s0_inv` 的贡献是：

```text
[c00*dp00, c01*dp01, c02*dp02, ..., c026*dp026]
```

这些贡献最后要合并回 `s0_inv` 这一个数：

```text
ds0_inv = c00*dp00 + c01*dp01 + c02*dp02 + ... + c026*dp026
```

代码就是：

```python
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
```

拆开理解：

```python
counts * dprobs
```

先得到每个位置的贡献表，shape 还是 `(32, 27)`。

```python
.sum(1, keepdim=True)
```

再把每一行的 27 列横向加起来，shape 从 `(32, 27)` 变回 `(32, 1)`，刚好和 `counts_sum_inv` 一样。

一句话规则：forward 里某个 `(32, 1)` 被 broadcast 成 `(32, 27)` 使用；backward 里就要沿着被 broadcast 出来的 27 列 `sum` 回 `(32, 1)`。


### `counts.sum(1)` 的 backward：把一个梯度分发回一整行

这里对应 forward：

```python
counts_sum = counts.sum(1, keepdims=True)
```

`counts` 是 `(32, 27)`，可以看成 32 行、27 列的表：

```text
              .       a       b      ...      z
样本0      c00     c01     c02     ...    c026
样本1      c10     c11     c12     ...    c126
...
样本31     c310    c311    c312    ...    c3126
```

forward 的 `sum(1)` 是每一行横向求和：

```text
样本0: s0 = c00 + c01 + c02 + ... + c026
样本1: s1 = c10 + c11 + c12 + ... + c126
...
```

所以：

```python
counts_sum.shape # (32, 1)
```

backward 时，从右边传回来：

```python
dcounts_sum.shape # (32, 1)
```

也就是每一行的 sum 收到一个梯度：

```text
样本0      ds0
样本1      ds1
...
样本31     ds31
```

因为：

```text
s0 = c00 + c01 + c02 + ... + c026
```

每个 `c0j` 对 `s0` 的影响都是 1，所以 `ds0` 要原样分发给第 0 行所有 27 个位置：

```text
第 0 行 dcounts = [ds0, ds0, ds0, ..., ds0]
```

代码：

```python
dcounts = torch.ones_like(counts) * dcounts_sum
```

拆开理解：

```python
torch.ones_like(counts)
```

先做一个 `(32, 27)` 的全 1 表；再乘上 `(32, 1)` 的 `dcounts_sum`。`dcounts_sum` 会 broadcast 到 `(32, 27)`，每一行自己的梯度复制到 27 列。

一句话规则：forward 的 `sum(1)` 把一行 27 个数合成 1 个数；backward 就把这 1 个梯度复制回这一行的 27 个位置。


### `db2 = dlogits.sum(0)`：为什么这里不用 `keepdim=True`

这里对应 forward：

```python
logits = h @ W2 + b2
```

shape 是：

```python
logits.shape # (32, 27)
b2.shape     # (27,)
```

`b2` 是每个输出类别一个 bias。forward 时它会 broadcast 到 32 个样本上：

```text
              27 个类别
样本0      + b2
样本1      + b2
样本2      + b2
...
样本31     + b2
```

所以 backward 时，`b2` 要把 32 个样本在每个类别上的梯度纵向加起来：

```python
db2 = dlogits.sum(0)
```

表格看：

```text
dlogits: (32, 27)

              .       a       b      ...      z
样本0      g00     g01     g02     ...    g026
样本1      g10     g11     g12     ...    g126
...
样本31     g310    g311    g312    ...    g3126
```

`sum(0)` 是沿 batch 维度纵向加：

```text
db2 = [
  g00 + g10 + ... + g310,
  g01 + g11 + ... + g311,
  g02 + g12 + ... + g312,
  ...,
  g026 + g126 + ... + g3126
]
```

所以：

```python
db2.shape # (27,)
```

这刚好和原变量一样：

```python
b2.shape # (27,)
```

如果写：

```python
db2 = dlogits.sum(0, keepdim=True)
```

shape 会变成：

```python
db2.shape # (1, 27)
```

数值类似，但 shape 和 `b2` 不一样，不适合直接用：

```python
cmp('b2', db2, b2)
```

判断规则：梯度的 shape 应该和它对应的原变量 shape 对齐。

```text
b2 是 (27,)        -> db2 用 sum(0)，得到 (27,)
counts_sum 是 (32,1) -> dcounts_sum 要 keepdim=True，保持 (32,1)
```


## Questions

- 
